In [ ]:
import ee
from lib.utils.ee.ccdc_utils import build_segment_tag, get_ccdc_coefs
from pprint import pprint

ee.Initialize()

numberOfStablePoints = 5
numberOfUnstablePoints = 10

aoi_bbox = [
    (-65.00330743296635, -9.986381612112117),
    (-65.00330743296635, -14.993821533992973),
    (-59.98255547984135, -14.993821533992973),
    (-59.98255547984135, -9.986381612112117),
]
aoi = ee.Geometry.Polygon(aoi_bbox)

startYear = 2016
endYear   = 2018

bands = ['SWIR1']
coefficients = ["INTP", "SLP", "COS", "SIN", "COS2", "SIN2", "COS3", "SIN3", "RMSE"]
segments = build_segment_tag(10)

ccdc_raw = ee.ImageCollection("GOOGLE/GLOBAL_CCDC/V1").mosaic()

def mid_decimal_year(y):
    return ee.Number(y).add(0.5)

coefs_start = get_ccdc_coefs(
    raw_ccdc_image=ccdc_raw,
    segs=segments,
    bands=bands,
    date=mid_decimal_year(startYear),
    coef_tags=coefficients,
    normalize=True,
    behavior="before"
).clip(aoi)

coefs_end = get_ccdc_coefs(
    raw_ccdc_image=ccdc_raw,
    segs=segments,
    bands=bands,
    date=mid_decimal_year(endYear),
    coef_tags=coefficients,
    normalize=True,
    behavior="before"
).clip(aoi)

delta = coefs_end.subtract(coefs_start).abs()
diffMag = delta.reduce(ee.Reducer.max())

noDiffMask = diffMag.eq(0)
diffMask   = diffMag.gt(0)

classImage = ee.Image(0).where(diffMask, 1).rename('class')

validMask = coefs_start.mask().reduce(ee.Reducer.allNonZero()) \
    .And(coefs_end.mask().reduce(ee.Reducer.allNonZero()))
sampleImg = classImage.addBands(diffMag.rename('diffMag')).updateMask(validMask)

# Get all valid points first, then sort by magnitude
allPoints = sampleImg.sample(
    region=aoi,
    scale=1000,
    geometries=True,
    seed=42
)

# Sort points by magnitude of difference (descending - highest first)
sortedPoints = allPoints.sort('diffMag', False)  # False = descending order

# Sample from the top points by magnitude
# Take top points for each class
stablePoints = sortedPoints.filter(ee.Filter.eq('class', 0)).limit(numberOfStablePoints)
unstablePoints = sortedPoints.filter(ee.Filter.eq('class', 1)).limit(numberOfUnstablePoints)

# Combine the sampled points
sampledPoints = stablePoints.merge(unstablePoints)

def add_props(f):
    coords = f.geometry().coordinates()
    status = ee.String(ee.Algorithms.If(ee.Number(f.get('class')).eq(1), 'changed', 'unchanged'))
    return f.set({
        'lon': coords.get(0),
        'lat': coords.get(1),
        'startYear': startYear,
        'endYear': endYear,
        'status': status
    })

samples_tagged = sampledPoints.map(add_props)

pprint(samples_tagged.getInfo())

# selectors = ['class', 'status', 'lon', 'lat', 'startYear', 'endYear', 'diffMag']
# task = ee.batch.Export.table.toDrive(
#     collection=samples_tagged,
#     description=f'ccdc_samples_{startYear}_{endYear}',
#     fileNamePrefix=f'ccdc_samples_{startYear}_{endYear}',
#     fileFormat='CSV',
#     selectors=selectors
# )
# task.start()

# print('Export started:', f'ccdc_samples_{startYear}_{endYear}')

{'columns': {'class': 'Byte<0, 1>',
             'diffMag': 'Float',
             'endYear': 'Integer',
             'lat': 'Object',
             'lon': 'Object',
             'startYear': 'Integer',
             'status': 'String'},
 'features': [{'geometry': {'coordinates': [-62.95842668751666,
                                            -13.299557781389517],
                            'geodesic': False,
                            'type': 'Point'},
               'id': '0',
               'properties': {'class': 0,
                              'diffMag': 0,
                              'endYear': 2018,
                              'lat': -13.299557781389517,
                              'lon': -62.95842668751666,
                              'startYear': 2016,
                              'status': 'unchanged'},
               'type': 'Feature'},
              {'geometry': {'coordinates': [-64.4047142949491,
                                            -12.158697370557723],
 

In [4]:
import json
import os
from datetime import datetime

samples_tagged_info = samples_tagged.getInfo()

# Extract points (longitude, latitude)
points_list = []
for feature in samples_tagged_info['features']:
    # Coordinates are typically [longitude, latitude] in GeoJSON
    coords = feature['geometry']['coordinates']
    points_list.append(coords)

# Format into the desired JSON structure
output_data = {"points": points_list}

# Construct the file path
current_datetime_str = datetime.now().strftime("%Y%m%d_%H%M%S")
total_points_count = numberOfStablePoints + numberOfUnstablePoints
group_dir_name = f"{current_datetime_str}-{total_points_count}_points"
output_base_dir = os.path.join("lib", "point_groups", "groups")
output_dir = os.path.join(output_base_dir, group_dir_name)
output_filepath = os.path.join(output_dir, "points.json")

# Create directories if they don't exist
os.makedirs(output_dir, exist_ok=True)

# Write the JSON file
with open(output_filepath, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f"Successfully saved {len(points_list)} points to: {output_filepath}")


Successfully saved 15 points to: lib/point_groups/groups/20250930_132850-15_points/points.json
